# Clean Dataset Pipeline

Очистка weekly-рядов запросов и сохранение в Parquet.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

In [ ]:
INPUT_DIRS = [
    Path(r"D:\\YandexDisk\\Гузель\\магистратура\\Диплом\\2021-03-02"),
    Path(r"D:\\YandexDisk\\Гузель\\магистратура\\Диплом\\2021-04-22"),
    Path(r"D:\\YandexDisk\\Гузель\\магистратура\\Диплом\\2021-04-24"),
]
OUT_DIR = Path(r"D:\\YandexDisk\\Гузель\\магистратура\\Диплом\\processed_parquet")
METRICS_DIR = OUT_DIR / "word_metrics"
FILTERED_WIDE_DIR = OUT_DIR / "filtered_wide"
LOG_PATH = OUT_DIR / "processed_files.jsonl"
GLOB_PATTERNS = ("*.txt.csv",)
CSV_KWARGS = dict(sep=";", encoding="utf-8", low_memory=False)
KEYWORD_COL = "Keyword"
MIN_NONZERO_WEEKS = 10
MIN_TOTAL_QUERIES: int | None = None
MAX_CV_NONZERO = None
MAX_SEASONALITY_52 = 0.6
DROP_WEEKS = ["2021-04-25", "2021-05-02", "2021-05-09"]
OUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FILTERED_WIDE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
DATE_HEADER_RE = re.compile(r"^\\d{4}-\\d{2}-\\d{2}$")
PART_NUM_RE = re.compile(r"(?i)part(\\d+)")

def _input_file_sort_key(p: Path) -> tuple:
    m = PART_NUM_RE.search(p.name)
    part_n = int(m.group(1)) if m else 10**12
    return (str(p.parent).lower(), part_n, p.name.lower())

def list_input_files(dirs: list[Path]) -> list[Path]:
    files: list[Path] = []
    for d in dirs:
        if not d.is_dir():
            continue
        for pat in GLOB_PATTERNS:
            files.extend(d.glob(pat))
    return sorted(set(files), key=_input_file_sort_key)

def parse_date_columns(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    keep = [KEYWORD_COL]
    dropped = []
    for c in df.columns:
        if c == KEYWORD_COL:
            continue
        s = str(c).strip()
        if DATE_HEADER_RE.match(s):
            keep.append(c)
        else:
            dropped.append(str(c))
    return df[keep].copy(), dropped

In [ ]:
def compute_all_metrics(df: pd.DataFrame, source_file: str) -> pd.DataFrame:
    cols = df.columns
    X = np.asarray(df.to_numpy(dtype=np.float64))
    n, T = X.shape
    keywords = df.index.astype(str).to_numpy()
    finite = np.isfinite(X)
    mask_pos = finite & (X > 0)
    nonzero_weeks = mask_pos.sum(axis=1).astype(np.int32)
    total_queries_pos = np.where(mask_pos, X, 0.0).sum(axis=1)
    total_queries_nonneg = np.where(finite & (X >= 0), X, 0.0).sum(axis=1)
    X_pos = np.where(mask_pos, X, np.nan)
    with np.errstate(all="ignore"):
        row_mean = np.nanmean(X_pos, axis=1)
    row_std = np.full(n, np.nan, dtype=np.float64)
    ge2 = nonzero_weeks >= 2
    if np.any(ge2):
        row_std[ge2] = np.nanstd(X_pos[ge2], axis=1, ddof=1)
    cv = row_std / row_mean
    cv = np.where(nonzero_weeks >= 2, cv, np.where(nonzero_weeks == 1, 0.0, np.nan))
    first_col = np.argmax(mask_pos, axis=1).astype(np.intp)
    sw = pd.Series(cols.take(first_col), index=df.index).where(mask_pos.any(axis=1), pd.NaT)
    if T >= 104:
        W = np.where(finite, X, 0.0)
        a, b = W[:, :-52], W[:, 52:]
        Ac = a - a.mean(axis=1, keepdims=True)
        Bc = b - b.mean(axis=1, keepdims=True)
        den = np.sqrt((Ac * Ac).sum(axis=1) * (Bc * Bc).sum(axis=1))
        with np.errstate(invalid="ignore", divide="ignore"):
            seas = np.divide((Ac * Bc).sum(axis=1), den, out=np.full(n, np.nan), where=den > 1e-30)
    else:
        seas = np.full(n, np.nan)
    keep = nonzero_weeks >= MIN_NONZERO_WEEKS
    if MIN_TOTAL_QUERIES is not None:
        keep &= total_queries_pos >= MIN_TOTAL_QUERIES
    if MAX_CV_NONZERO is not None:
        keep &= (~np.isfinite(cv)) | (cv <= MAX_CV_NONZERO)
    if MAX_SEASONALITY_52 is not None:
        keep &= (~np.isfinite(seas)) | (np.abs(seas) <= MAX_SEASONALITY_52)
    return pd.DataFrame({
        "nonzero_weeks": nonzero_weeks,
        "total_queries_pos": total_queries_pos,
        "total_queries_nonneg": total_queries_nonneg,
        "cv_nonzero": cv,
        "seasonality_52": seas,
        "start_week": sw.values,
        "keyword": keywords,
        "source_file": source_file,
        "keep": keep,
    })

In [ ]:
def process_one_csv(path: Path, write_outputs: bool = True) -> dict:
    df = pd.read_csv(path, **CSV_KWARGS)
    if KEYWORD_COL not in df.columns:
        raise ValueError(f"Нет колонки {KEYWORD_COL}: {path}")
    df, dropped_cols = parse_date_columns(df)
    week_cols = [c for c in df.columns if c != KEYWORD_COL]
    df[week_cols] = df[week_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=[KEYWORD_COL])
    df[KEYWORD_COL] = df[KEYWORD_COL].astype(str).str.strip()
    df = df[df[KEYWORD_COL] != ""]
    num = df[week_cols]
    df = df.loc[~((num.isna()) | (num == 0)).all(axis=1)].copy()
    if df.empty:
        return {"file": str(path), "rows_metrics": 0, "rows_kept": 0, "dropped_cols": dropped_cols}
    df = df.set_index(KEYWORD_COL)
    df.columns = pd.to_datetime(df.columns, errors="coerce")
    df = df.drop(columns=df.columns[df.columns.isna()])
    df = df.sort_index(axis=1)
    drop_idx = pd.to_datetime(DROP_WEEKS, errors="coerce")
    df = df.drop(columns=df.columns.intersection(drop_idx[~pd.isna(drop_idx)]), errors="ignore")
    metrics = compute_all_metrics(df, str(path))
    df_filt = df.loc[df.index.isin(set(metrics.loc[metrics["keep"], "keyword"]))]
    if write_outputs:
        safe = f"{path.parent.name}_{path.stem.replace('.', '_')}"
        metrics_path = METRICS_DIR / f"{safe}.parquet"
        wide_path = FILTERED_WIDE_DIR / f"{safe}.parquet"
        metrics.to_parquet(metrics_path, index=False)
        df_filt.to_parquet(wide_path)
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({
                "file": str(path),
                "rows_metrics": int(len(metrics)),
                "rows_kept": int(metrics["keep"].sum()),
                "metrics_parquet": str(metrics_path),
                "wide_parquet": str(wide_path),
                "dropped_non_date_cols": dropped_cols,
            }, ensure_ascii=False) + "\n")
    return {
        "file": str(path),
        "rows_metrics": int(len(metrics)),
        "rows_kept": int(metrics["keep"].sum()),
        "dropped_cols": dropped_cols,
    }